# 19장 · LCEL 체인과 출력 파서

LCEL의 파이프 연산자(|)는 앞 구성 요소의 출력을 다음 구성 요소의 입력으로 넘깁니다. 프롬프트, 모델, 출력 파서가 각각 주고받는 값이 다릅니다.

도서관 안내 문장과 질문을 LCEL 프롬프트에 전달합니다. 이어지는 모델과 출력 파서의 입력·반환 타입을 확인하고, 스트리밍 조각에서 전체 문자열을 복원합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## ChatPromptTemplate의 입력 변수

템플릿은 비어 있는 칸에 이번 자료를 넣어 요청을 만드는 틀입니다. context에는 근거를, question에는 묻는 문장을 넣습니다. 먼저 표준 파이썬으로 그 두 값을 확인합니다.


### 입력 키와 자료형 오류

<table><caption>증상과 확인</caption><thead><tr><th scope="col">증상</th><th scope="col">먼저 확인</th></tr>
</thead><tbody><tr><td>context 키 오류</td><td>템플릿의 변수 이름과 입력 키</td></tr>
<tr><td>AIMessage를 문자열처럼 처리</td><td>파서 연결 여부와 반환 타입</td></tr>
<tr><td>질문을 바꿔도 같은 답</td><td>FakeListChatModel의 고정 응답</td></tr>
<tr><td>모듈을 찾을 수 없음</td><td>현재 가상환경과 requirements</td></tr>
<tr><td>stream이 한꺼번에 보임</td><td>구성 요소의 스트리밍 지원·출력 처리</td></tr>
</tbody></table>


### 셀 01 · 프롬프트 입력 딕셔너리

context에는 근거 문장, question에는 사용자 질문을 전달합니다.


In [1]:
inputs = {"context": "수업용 가상 도서관의 대출 기간은 14일이다.", "question": "대출은 며칠인가요?"}
print(inputs["context"])

수업용 가상 도서관의 대출 기간은 14일이다.


프롬프트의 변수명 context는 입력 사전의 키와 일치해야 합니다.

여기서 질문만 ‘휴관일은 언제인가요?’로 고쳐 보세요. 질문 칸은 바뀌지만 context에는 여전히 대출 기간이 들어 있습니다. 템플릿에 값을 잘 채우는 것과 질문에 맞는 근거를 찾는 것은 서로 다른 작업입니다. 23장에서 질문에 맞는 문서를 찾고 24장에서 그 결과를 체인의 문맥으로 넘깁니다.


### 프롬프트 변수와 입력 키

<p>context는 모델에게 제공할 사실이고 question은 사용자의 질문입니다. 템플릿에 {context}와 {question}을 사용했으므로 invoke 입력에도 두 키가 있어야 합니다. 이름이 다르거나 빠지면 모델을 호출하기 전에 오류가 날 수 있습니다.</p>

<p>system 메시지에 넣는 자료도 자동으로 사실이 되는 것은 아닙니다. 이 예제는 명시된 가상 자료만 사용합니다. 이후 문서 검색 수업에서 context를 직접 작성하는 대신 검색한 문서로 만들 것입니다.</p>


### 과제 02 · context와 question 치환

파이썬 문자열 format에 inputs를 전달해 근거와 질문이 각각 들어간 문장을 출력하세요. 이 셀은 LCEL을 쓰기 전의 작은 연습입니다.


In [2]:
# {context}와 {question}을 실제 값으로 바꾸세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>template = &quot;근거: {context}\n질문: {question}&quot;
print(template.format(**inputs))</code></pre>
<p>실행 결과</p><pre><code>근거: 수업용 가상 도서관의 대출 기간은 14일이다.
질문: 대출은 며칠인가요?</code></pre>
<p>첫 줄에는 자료가, 둘째 줄에는 질문이 남습니다. 템플릿 자체를 화면에 찍으면 중괄호 안의 이름이 그대로 보입니다.</p>
</details>


### 프롬프트 입력 치환 실습

<p>다음 장부터는 모델에 제공할 근거 문서를 준비합니다. LCEL 문법을 더 늘리기보다 문서·검색·근거 답변의 흐름에 집중합니다.</p>


## 파이프 연산자와 StrOutputParser, invoke()

LangChain Expression Language(LCEL)의 |는 앞 구성 요소가 돌려준 값을 다음 요소에 건넵니다. 수업 환경에 langchain-core가 설치되어 있으면 아래 셀로 실제 체인을 만듭니다. FakeListChatModel은 고정 답변을 내는 수업용 모델입니다.

**LCEL의 연속 변환**　카드가 첫 필터를 통과한 뒤 두 번째 필터로 들어갑니다. 코드에서 각 단계가 반환하는 메시지와 문자열을 차례로 확인하세요.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter19/index.html#concept-clip)


### LCEL 구성 요소의 입출력

<p>LCEL은 LangChain 구성 요소의 입출력을 연결하는 표현 방식입니다. Runnable은 invoke 같은 공통 실행 방법으로 사용할 수 있는 구성 요소입니다. 이번 장에서는 복잡한 추상화보다 프롬프트, 모델, 출력 파서 세 개만 연결합니다.</p>

<p>prompt | model | parser에서 |는 문자열을 합치는 기호가 아니라 구성 요소를 순서대로 연결하는 연산입니다. 한 요소의 출력이 다음 요소의 입력으로 들어갑니다. 13장에서 Python 함수로 만든 순차 흐름과 같은 입출력 관점으로 읽으세요.</p>

<table><caption>연결 순서</caption><thead><tr><th scope="col">구성 요소</th><th scope="col">입력</th><th scope="col">출력</th></tr>
</thead><tbody><tr><td>ChatPromptTemplate</td><td>변수가 있는 딕셔너리</td><td>메시지를 담은 프롬프트 값</td></tr>
<tr><td>채팅 모델</td><td>메시지 프롬프트</td><td>AI 메시지</td></tr>
<tr><td>StrOutputParser</td><td>AI 메시지</td><td>문자열</td></tr>
</tbody></table>


### 셀 03 · to_messages()의 content

모델을 부르기 전에 ChatPromptTemplate이 만든 값에서 질문을 먼저 꺼내 봅니다.


In [3]:
from importlib.util import find_spec
has_lcel = find_spec("langchain_core") is not None
if has_lcel:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.language_models.fake_chat_models import FakeListChatModel
    from langchain_core.output_parsers import StrOutputParser
    prompt = ChatPromptTemplate.from_messages([("system", "주어진 사실만 사용하세요. 사실: {context}"),
                                               ("human", "{question}")])
    print(prompt.invoke(inputs).to_messages()[-1].content)
else:
    print("langchain-core가 설치된 수업 커널에서 메시지 변환을 실행하세요.")

대출은 며칠인가요?


마지막 메시지에는 ‘대출은 며칠인가요?’라는 질문이 들어 있습니다. prompt의 반환값 안에는 system 메시지도 들어 있습니다. 둘을 합친 메시지 묶음이 모델 입력입니다.


### 구성 요소별 중간 반환값

<p>ChatPromptValue는 메시지를 담은 프롬프트 값이며 to_messages()로 system과 human 메시지를 볼 수 있습니다. 채팅 모델의 AIMessage에는 content가 있습니다. StrOutputParser를 거친 뒤에는 문자열 자체를 사용합니다. 설치한 1.8.0에서 표시되는 타입명은 TextAccessor이지만 str의 하위 타입이므로 isinstance(parsed, str)는 True이고 문자열처럼 사용합니다.</p>

<p>이 문자열 파서는 12일에 배운 JSON 스키마 검사나 내용 검증을 수행하지 않습니다. 18일의 입력 검사 책임도 사라지지 않습니다. 모델이 돌려준 문장이 사실인지 확인하려면 별도 근거와 평가가 필요합니다.</p>


### 셀 04 · StrOutputParser와 invoke()

고정 모델 뒤에 문자열 파서를 이어 실제 LCEL 경로를 실행합니다.

모델 뒤에서 멈추면 답은 메시지 객체에 담겨 있습니다. 뒤에 StrOutputParser를 연결하면 invoke의 반환값으로 본문 문자열을 얻습니다. 이 노트북의 실행 기록은 langchain-core 1.6.12 기준이고 더 읽기의 1.8.0은 별도 기록입니다.


In [4]:
if has_lcel:
    model = FakeListChatModel(responses=["가상 도서관의 대출 기간은 14일입니다."])
    chain = prompt | model | StrOutputParser()
    print(chain.invoke(inputs))
else:
    print("LCEL 체인을 실행하려면 langchain-core가 필요합니다.")

가상 도서관의 대출 기간은 14일입니다.


체인이 실행되면 ‘가상 도서관의 대출 기간은 14일입니다.’라는 문자열이 나옵니다. 파서는 모델이 만든 메시지에서 글을 꺼내고, 지정한 고정 모델은 원문의 사실 여부를 검증하지 않습니다.


### 고정 모델을 이용한 체인 검증

<p>FakeListChatModel은 지정한 응답을 반환하는 테스트용 모델입니다. 여기서는 실제 LLM 서버나 API 키 없이 체인과 파서가 연결되는지 확인하기 위해 사용합니다. 질문을 바꿔도 준비된 답이 나올 수 있으므로 답변 정확도 평가에는 사용할 수 없습니다.</p>

<p>실제 모델을 연결할 때는 같은 채팅 모델 인터페이스를 지원하는 구성 요소로 바꿉니다. 계정 모델·지원 옵션·버전·비용을 확인하고 응답 결과를 별도로 검증해야 합니다.</p>


### invoke() 단일 호출

<p>invoke의 결과는 StrOutputParser를 거친 문자열입니다. 파서를 빼고 모델까지만 연결하면 메시지 객체를 받습니다. 문자열과 메시지 객체의 차이를 출력 타입으로 확인하면 뒤 단계에서 잘못된 필드를 읽는 실수를 줄일 수 있습니다.</p>


## stream()과 출력 조각 결합

invoke는 한 번에 결과를 받고 stream은 차례로 오는 조각을 읽습니다. 조각 하나의 경계는 문장이나 단어의 경계와 같지 않을 수 있습니다.


### 과제 05 · 스트림 문자열 결합

LCEL 환경에서는 chain.stream의 조각을 이어 전체 답을 출력하세요. 라이브러리가 없는 경우 제공 조각으로 결합 연산만 연습합니다.


In [5]:
pieces = ["가상 ", "안내"]
# 수업 환경에 맞게 출력 조각을 이어 보세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>if has_lcel:
    pieces = chain.stream(inputs)
else:
    pieces = [&quot;가상 &quot;, &quot;안내&quot;]
print(&quot;&quot;.join(pieces))</code></pre>
<p>실행 결과</p><pre><code>가상 도서관의 대출 기간은 14일입니다.</code></pre>
<p>실제 체인을 쓴 경우 앞의 invoke와 같은 고정 문장이 됩니다. 제공 조각만 썼다면 ‘가상 안내’가 결과입니다. 고정 모델의 조각 수를 실제 API의 토큰 수로 해석할 수는 없습니다.</p>
</details>


### 조각을 도착 순서대로 이어 붙이기

<p>pieces에는 ‘가상 ’과 ‘안내’ 두 문자열이 들어 있습니다. 빈 문자열로 join하면 첫 조각 끝에 들어 있던 공백을 보존한 ‘가상 안내’가 됩니다. 이 셀은 조각을 다시 합치는 연습으로, 실제 모델이 언제 어떤 길이로 보냈는지는 측정하지 않습니다.</p>


## 돌아보기

입력 사전, 모델에 보낼 메시지 묶음, 마지막 문자열을 순서대로 가리켜 보세요. 다음 장에서는 context를 손으로 쓰는 대신 어느 문서에서 가져올지 정합니다.
